# 🎯 MODEL 3: Phân loại Nguyên nhân Gốc rễ Đa lớp bằng XGBoost (XGBoost Multi-class RCA Classifier)

---
### 📌 Cơ chế Tự Chọn Đặc trưng từ Unified Dataset (`aiops_network_dataset.csv`):
- **File Dataset dùng chung cho cả 3 Model:** `aiops_network_dataset.csv` (`700 mẫu × 117 cột`).
- **Thuật toán chính thức:** **XGBoost (`XGBClassifier` - Extreme Gradient Boosting)** với hàm mục tiêu `multi:softprob` / `mlogloss`.
- **Tập đặc trưng Model 3 tự chọn (`42 đặc trưng trạng thái tức thời đa tầng`):** Tự động lọc bỏ các cột cửa sổ trượt (`_roll_mean`, `_roll_std`, `_delta1`, `_trend_slope`, `_sustained_ratio`) để lấy đúng **42 thông số kỹ thuật hiện tại** ở 4 tầng (WAN1/WAN2, Firewall, LAN/TCP Stack, Nginx/Flask API/PostgreSQL).
- **Cột Nhãn Mục tiêu (`Target`) Model 3 tự chọn:** `label` (7 lớp: `Normal`, `WanCongestion`, `WanLossLatency`, `SynFlood`, `PortScan`, `SshBruteforce`, `HttpFlood`).
- **File Model xuất ra:** `model3_rca_classifier.pkl`

In [ ]:
import glob
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (11, 6)

## 1. Đọc Unified Dataset (`aiops_network_dataset.csv`) & Tự Chọn 42 Đặc trưng Đa tầng

In [ ]:
candidates = (
    glob.glob('/kaggle/input/**/aiops_network_dataset.csv', recursive=True)
    + ['../../dataset/aiops_network_dataset.csv', '../dataset/aiops_network_dataset.csv', 'aiops_network_dataset.csv']
)
csv_path = next((p for p in candidates if os.path.exists(p)), None)
assert csv_path is not None, 'Hãy upload file aiops_network_dataset.csv lên Kaggle!'

df_all = pd.read_csv(csv_path).replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Lọc bỏ các dòng gai nhiễu thoáng qua (Normal nhưng có current_degraded=1) để Model 3 học thuần khiết chữ ký 7 lớp
df_m3 = df_all[~((df_all['label'] == 'Normal') & (df_all['current_degraded'] == 1))].reset_index(drop=True)

META_AND_TARGETS = {'timestamp', 'datetime', 'is_anomaly', 'current_degraded', 'degraded_forecast_5m', 'label'}
ROLLING_SUFFIXES = ('_roll_mean', '_roll_std', '_delta1', '_trend_slope', '_sustained_ratio')

# Model 3 tự động chọn 42 đặc trưng trạng thái tức thời đa tầng
MODEL3_FEATURES = [
    c for c in df_m3.columns
    if c not in META_AND_TARGETS and not c.endswith(ROLLING_SUFFIXES)
]
TARGET_COL = 'label'

print(f'✅ Đã nạp Unified Dataset: {csv_path}')
print(f'🎯 Model 3 tự động chọn : {len(MODEL3_FEATURES)} đặc trưng đa tầng ({len(df_m3)} mẫu chuẩn) | Target: {TARGET_COL}')
display(df_m3[['datetime', *MODEL3_FEATURES[:6], TARGET_COL]].head())

## 2. Trực quan hóa Không gian 42 Đặc trưng bằng PCA 2D & t-SNE 2D

In [ ]:
X_vis = StandardScaler().fit_transform(df_m3[MODEL3_FEATURES].values)
pca_2d = PCA(n_components=2, random_state=42).fit_transform(X_vis)
tsne_2d = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_vis)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(x=pca_2d[:, 0], y=pca_2d[:, 1], hue=df_m3[TARGET_COL], style=df_m3[TARGET_COL], s=75, palette='tab10', ax=ax[0])
ax[0].set_title('PCA 2D Projection - 42 Đặc trưng Trạng thái Mạng & Dịch vụ', fontweight='bold')

sns.scatterplot(x=tsne_2d[:, 0], y=tsne_2d[:, 1], hue=df_m3[TARGET_COL], style=df_m3[TARGET_COL], s=75, palette='tab10', ax=ax[1])
ax[1].set_title('t-SNE 2D Clustering - Phân tách 7 Lớp Sự cố Rõ rệt', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Huấn luyện & Đối sánh Thuật toán Phân loại Đa lớp (Model 3)

In [ ]:
X = df_m3[MODEL3_FEATURES].values
le = LabelEncoder()
y = le.fit_transform(df_m3[TARGET_COL].values)
class_names = list(le.classes_)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)
X_all_s = StandardScaler().fit_transform(X)

candidates = {
    'XGBoost_RCA': XGBClassifier(n_estimators=300, learning_rate=0.06, max_depth=4, subsample=0.95, colsample_bytree=0.95, reg_alpha=0.05, reg_lambda=1.0, eval_metric='mlogloss', random_state=42, n_jobs=-1),
    'RandomForest_RCA': RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1),
    'ExtraTrees_RCA': ExtraTreesClassifier(n_estimators=300, max_depth=14, class_weight='balanced', random_state=42, n_jobs=-1),
    'GradientBoosting_RCA': GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=4, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
trained_models = {}

for name, clf in candidates.items():
    clf.fit(X_tr_s, y_tr)
    preds = clf.predict(X_te_s)
    cv_scores = cross_val_score(clf, X_all_s, y, cv=cv, scoring='f1_macro', n_jobs=-1)
    trained_models[name] = (clf, preds)
    results.append({
        'Model 3 Algorithm': name,
        'Test Accuracy': accuracy_score(y_te, preds),
        'Macro Precision': precision_score(y_te, preds, average='macro'),
        'Macro Recall': recall_score(y_te, preds, average='macro'),
        'Macro F1-Score': f1_score(y_te, preds, average='macro'),
        '5-Fold CV F1': np.mean(cv_scores),
    })

res_df = pd.DataFrame(results)
display(res_df.round(4))

In [ ]:
deployed_name = 'XGBoost_RCA'
xgb_clf, xgb_preds = trained_models[deployed_name]

print(f'🏆 Mô hình Phân loại Nguyên nhân Sự cố (Model 3) triển khai chính thức: {deployed_name}')
print(classification_report(y_te, xgb_preds, target_names=class_names, digits=4))

cm = confusion_matrix(y_te, xgb_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix - Model 3 ({deployed_name})', fontweight='bold')
plt.xlabel('Nhãn Dự đoán (Predicted Fault)')
plt.ylabel('Nhãn Thực tế (True Fault)')
plt.tight_layout()
plt.show()

imp_df = pd.DataFrame({'Feature': MODEL3_FEATURES, 'Importance': xgb_clf.feature_importances_}).sort_values('Importance', ascending=False).head(15)
plt.figure(figsize=(11, 6))
sns.barplot(x='Importance', y='Feature', data=imp_df, palette='magma')
plt.title(f'Top 15 Đặc trưng Quyết định Phân loại Nguyên nhân Sự cố (Model 3: {deployed_name})', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Đóng gói & Xuất File `model3_rca_classifier.pkl` (XGBoost)

In [ ]:
final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X)

final_clf = XGBClassifier(
    n_estimators=300,
    learning_rate=0.06,
    max_depth=4,
    subsample=0.95,
    colsample_bytree=0.95,
    reg_alpha=0.05,
    reg_lambda=1.0,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)
final_clf.fit(X_all_scaled, y)

feat_imp_pairs = sorted(
    zip(MODEL3_FEATURES, [round(float(x), 4) for x in final_clf.feature_importances_]),
    key=lambda item: item[1],
    reverse=True,
)

artifact_m3 = {
    'model_type': 'XGBoost_RCA',
    'model': final_clf,
    'scaler': final_scaler,
    'label_encoder': le,
    'class_names': class_names,
    'feature_names': MODEL3_FEATURES,
    'feature_importances': dict(feat_imp_pairs),
}

joblib.dump(artifact_m3, 'model3_rca_classifier.pkl')
print('✅ Đã xuất thành công Model 3 (XGBoost_RCA): model3_rca_classifier.pkl')